# Project Bài 7 · Có nên thêm tỷ lệ trễ của người bán vào cảnh báo lúc duyệt đơn?

**Câu hỏi.** Hệ thống cảnh báo đơn trễ của Bài 5 dùng sáu đặc trưng lúc duyệt. Đồng nghiệp đề xuất thêm r, tỷ lệ trễ lịch sử của người bán. Hãy trả lời bằng một đánh giá trung thực.

**Dữ liệu.** `data/olist_extended_v3.csv.gz` (Olist, Kaggle): mọi đơn đủ điều kiện duyệt 01/2017–03/2018, nhãn cuối cùng.
- **Phát triển:** 43.345 đơn duyệt năm 2017; 8 fold cuốn theo tháng T5–T12/2017 như slide của bài.
- **Tập cuối:** 20.608 đơn duyệt 01–03/2018, chỉ chấm qua `cham_cuoi`, **đúng một lần**. Giai đoạn này nằm trong test lịch sử của Olist, nhưng chưa từng được dùng để chọn logistic + r.

**Ba mức.** Cơ bản (5 điểm): bước 1–4. Khá (3 điểm): bước 5–6. Nâng cao (2 điểm): chọn một trong A, B, C ở cuối.
**Rubric 10 điểm.** Tái lập 3 · cài đặt và đối chiếu 3 · đánh giá 2 · diễn giải và giới hạn 2. Chấm cách làm và cách đọc kết quả, không chấm con số AP cao hay thấp.
**Ba mốc nộp.** M1: bước 1–2 · M2: bước 3–4 · M3: bước 5–6 và báo cáo.

Ô có `TODO` là việc của bạn; ô `TỰ KIỂM` in ✅ khi đúng. Chạy từ trên xuống. **Không** đọc `_CUOI` trực tiếp.

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import json, time, hashlib
from fractions import Fraction
from math import comb
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import average_precision_score

ROOT = Path.cwd()
TEP = ROOT / 'data' / 'olist_extended_v3.csv.gz'
assert TEP.exists(), 'Mở notebook từ thư mục bài (có data/olist_extended_v3.csv.gz).'
assert hashlib.sha256(TEP.read_bytes()).hexdigest() == 'ce93d20c8b46f984d607c79a2751cfb8c165285351544847d2a6e2e8e85fa9e3', 'sai bản lưu dữ liệu'
DATES = ['order_approved_at', 'order_delivered_carrier_date', 'order_estimated_delivery_date', 'order_delivered_customer_date', 'review_creation_date']
BANG = pd.read_csv(TEP, dtype={'period': str}, parse_dates=DATES)
PHAT_TRIEN = BANG[BANG.period == '2017'].reset_index(drop=True)      # dữ liệu phát triển: đơn duyệt năm 2017
_CUOI = BANG[BANG.period == '2018Q1'].reset_index(drop=True)          # tập cuối: CHỈ dùng qua cham_cuoi
FEATURES = ['distance_km', 'promised_days', 'price', 'freight_value', 'n_items', 'weight_g']
THANG = ['2017-05', '2017-06', '2017-07', '2017-08', '2017-09', '2017-10', '2017-11', '2017-12']
M_LAM_TRON = 10

def LOG():
    # logistic đã khoá của Bài 5: trung vị → StandardScaler → LogisticRegression(C = 1, newton-cg)
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('lr', LogisticRegression(C=1, solver='newton-cg', max_iter=100))])

def ap(y, s):
    return float(average_precision_score(np.asarray(y).astype(int), s))

def kiem_dinh_dau(thang, n):
    # kiểm định dấu hai phía chính xác (đã bỏ hoà): p = 2 · P(X ≥ max(thắng, thua)), X ~ Nhị thức(n, 1/2)
    if n == 0:
        return Fraction(1)
    k = max(thang, n - thang)
    return min(Fraction(1), 2 * Fraction(sum(comb(n, j) for j in range(k, n + 1)), 2 ** n))

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

_DAU_VET = ROOT / '.project_bai07_cham_cuoi.json'
def _van_tay(quy_tac, chon, tao_dac_trung):
    # Dấu vân tay của lần chấm: quy tắc đã viết + lựa chọn + dự đoán của LOG_R trên fold phát triển T11.
    # Đổi quy tắc, đổi lựa chọn, hay giữ nguyên chuỗi mà đổi hàm tạo đặc trưng (đổi mô hình) đều làm dấu vân tay khác.
    moc = pd.Timestamp('2017-11-01')
    hoc, cham = PHAT_TRIEN[PHAT_TRIEN.order_delivered_customer_date < moc], PHAT_TRIEN[PHAT_TRIEN.month == '2017-11']
    Xh, Xc = tao_dac_trung(hoc, cham)
    s = LOG().fit(Xh, hoc.late).predict_proba(Xc)[:, 1]
    du_doan = hashlib.sha256(np.round(s, 6).tobytes()).hexdigest()      # làm tròn 6 chữ số: ổn định khi chạy lại trên cùng môi trường
    return dict(quy_tac=quy_tac, chon=chon, du_doan_T11=du_doan, van_tay=hashlib.sha256((quy_tac + '|' + chon + '|' + du_doan).encode('utf-8')).hexdigest())

def cham_cuoi(quy_tac, chon, tao_dac_trung):
    # Chấm tập cuối đúng MỘT lần. Học trên mọi đơn 2017 đã giao trước 01/01/2018; chấm mọi đơn duyệt 01–03/2018 (nhãn cuối cùng).
    # quy_tac: chuỗi quy tắc chọn đã viết TRƯỚC khi chấm; chon ∈ {'LOG', 'LOG_R'}; tao_dac_trung(hoc, cham) trả (X_hoc, X_cham) cho LOG_R.
    # Luôn báo kèm LOG để so theo cặp. Lần gọi đầu lưu quy tắc và dấu vân tay; gọi lại y hệt nhận lại kết quả cũ; khác đi sẽ bị chặn.
    # Đây là cơ chế nhắc kỷ luật, không phải bảo mật: giảng viên đối chiếu tệp dấu vết khi chấm.
    vt = _van_tay(quy_tac, chon, tao_dac_trung)
    if _DAU_VET.exists():
        cu = json.loads(_DAU_VET.read_text(encoding='utf-8'))
        if cu['van_tay'] != vt['van_tay']:
            khac = [ten for ten, k in (('quy tắc', 'quy_tac'), ('lựa chọn', 'chon'), ('dự đoán trên fold phát triển (hàm tạo đặc trưng / mô hình)', 'du_doan_T11')) if cu[k] != vt[k]]
            raise RuntimeError(f"Tập cuối đã được chấm lúc {cu['luc']} (chọn {cu['chon']}). Lần này khác ở: {', '.join(khac)}. Không chấm lại.")
        return cu['ket_qua']
    hoc = PHAT_TRIEN[PHAT_TRIEN.order_delivered_customer_date < pd.Timestamp('2018-01-01')]
    s_log = LOG().fit(hoc[FEATURES], hoc.late).predict_proba(_CUOI[FEATURES])[:, 1]
    Xh, Xc = tao_dac_trung(hoc, _CUOI)
    s_r = LOG().fit(Xh, hoc.late).predict_proba(Xc)[:, 1]
    kq = dict(n=len(_CUOI), ty_le=float(_CUOI.late.mean()), n_hoc=len(hoc), ap_LOG=ap(_CUOI.late, s_log), ap_LOG_R=ap(_CUOI.late, s_r), chon=chon, theo_thang=[])
    for t, g in _CUOI.groupby('month'):
        i = g.index.to_numpy()
        kq['theo_thang'].append(dict(thang=t, n=len(g), ty_le=float(g.late.mean()), ap_LOG=ap(g.late, s_log[i]), ap_LOG_R=ap(g.late, s_r[i])))
    _DAU_VET.write_text(json.dumps(dict(vt, ket_qua=kq, luc=time.strftime('%Y-%m-%d %H:%M')), ensure_ascii=False), encoding='utf-8')
    return kq
print('phát triển (2017):', len(PHAT_TRIEN), 'đơn · tập cuối (01–03/2018):', len(_CUOI), 'đơn, chưa mở')

## Bước 1 (cơ bản) · Kiểm cột theo thời điểm duyệt

In [ ]:
COT_KIEM = ['price', 'freight_value', 'n_items', 'weight_g', 'distance_km', 'promised_days', 'order_estimated_delivery_date', 'seller_id', 'customer_unique_id', 'order_delivered_carrier_date', 'carrier_days', 'order_delivered_customer_date', 'delivery_days', 'review_score', 'review_creation_date']
# TODO: với dự đoán lúc DUYỆT đơn, gán 'có' hoặc 'chưa' cho từng cột trong COT_KIEM (dùng bảng kiểm bốn câu của bài).
#       Ghi các giả định của bạn vào GIA_DINH (chuỗi, ít nhất một câu).
LUC_DUYET = {c: ... for c in COT_KIEM}
GIA_DINH = '...'
raise NotImplementedError

In [ ]:
# TỰ KIỂM · bước 1
kiem('đủ 15 cột, mỗi cột “có” hoặc “chưa”', set(LUC_DUYET) == set(COT_KIEM) and set(LUC_DUYET.values()) <= {'có', 'chưa'})
kiem('sáu đặc trưng của Bài 5 đều có lúc duyệt', all(LUC_DUYET[c] == 'có' for c in FEATURES))
kiem('ngày giao thật và số ngày giao: chưa có lúc duyệt', LUC_DUYET['order_delivered_customer_date'] == LUC_DUYET['delivery_days'] == 'chưa', 'nhãn được tính từ ngày giao thật')
kiem('điểm đánh giá và ngày tạo đánh giá: chưa có lúc duyệt', LUC_DUYET['review_score'] == LUC_DUYET['review_creation_date'] == 'chưa', 'khách chấm sau khi nhận hàng hoặc tới ngày hẹn')
kiem('ngày giao hãng: chưa có lúc duyệt', LUC_DUYET['carrier_days'] == LUC_DUYET['order_delivered_carrier_date'] == 'chưa', 'chỉ có khi người bán giao cho hãng')
kiem('mã người bán có lúc duyệt (đặc trưng r dựa vào nó)', LUC_DUYET['seller_id'] == 'có')
kiem('đã ghi giả định', isinstance(GIA_DINH, str) and len(GIA_DINH) > 20)

## Bước 2 (cơ bản) · Chia theo tháng và dựng r chỉ từ phần học

In [ ]:
def chia_thang(bang, thang):
    # TODO: trả (hoc, cham): hoc = đơn đã giao TRƯỚC ngày 1 của tháng `thang` ('2017-05' …); cham = mọi đơn duyệt trong tháng đó (cột month)
    raise NotImplementedError

def ty_le_nguoi_ban(hoc, m=M_LAM_TRON):
    # TODO: trả (rate, p0): rate[seller] = (số đơn trễ + m·p0)/(số đơn + m) tính trên `hoc`; p0 = tỷ lệ trễ của `hoc`
    raise NotImplementedError

def gan_r(bang, da_hoc):
    # TODO: trả mảng r cho từng đơn của `bang`; người bán chưa có trong phần học nhận p0
    raise NotImplementedError

def r_cheo(hoc, m=M_LAM_TRON):
    # TODO: r của CHÍNH các dòng học, tính chéo: StratifiedKFold(5, shuffle=True, random_state=7) trên nhãn của `hoc`;
    #       với mỗi (a, b): r của các dòng b học từ các dòng a (ty_le_nguoi_ban rồi gan_r). Trả mảng dài len(hoc).
    raise NotImplementedError

def tao_dac_trung(hoc, cham, m=M_LAM_TRON):
    # TODO: trả (X_hoc, X_cham) = sáu đặc trưng + cột r. Dòng học: r_cheo(hoc, m). Dòng chấm: r học từ toàn bộ `hoc`.
    raise NotImplementedError

In [ ]:
# TỰ KIỂM · bước 2
co = [(len(h), len(c)) for h, c in (chia_thang(PHAT_TRIEN, t) for t in THANG)]
kiem('số đơn phần học và phần chấm của 8 fold khớp slide', co == [(5850, 3547), (9601, 3143), (12824, 3827), (16278, 4217), (20580, 4162), (24537, 4441), (29031, 7149), (33701, 5675)], 'phần học: order_delivered_customer_date < ngày 1 của tháng')
hoc, cham = chia_thang(PHAT_TRIEN, '2017-11'); rate, p0 = ty_le_nguoi_ban(hoc)
kiem('p̄ của phần học fold T11 = 941/29031', abs(p0 - 941 / 29031) < 1e-15)
s0 = hoc.seller_id.value_counts().index[0]; g0 = hoc[hoc.seller_id == s0]
kiem('r của người bán nhiều đơn nhất đúng công thức', abs(rate[s0] - (g0.late.sum() + 10 * p0) / (len(g0) + 10)) < 1e-12)
r_cham = gan_r(cham, (rate, p0)); moi = ~cham.seller_id.isin(rate)
kiem('người bán chưa gặp nhận p̄', np.allclose(r_cham[moi.to_numpy()], p0) and moi.any())
Xh, Xc = tao_dac_trung(hoc, cham)
kiem('tao_dac_trung: 7 cột, r của dòng chấm học từ phần học', list(Xh.columns) == FEATURES + ['r'] and np.allclose(Xc.r, r_cham))
mot = hoc.seller_id.map(hoc.seller_id.value_counts()).eq(1).to_numpy()
kiem('dòng học tính chéo: người bán 1 đơn nhận ≈ p̄, không phụ thuộc nhãn của chính nó', mot.any() and np.abs(Xh.r.to_numpy()[mot] - p0).max() < 0.002,
     'r của dòng học phải học từ các phần KHÁC của phần học (r_cheo), không tính trên chính phần học')
kiem('dòng học tính chéo: r khác r tính trong mẫu', not np.allclose(Xh.r, gan_r(hoc, (rate, p0))))

**Câu hỏi.** Một người bán có 1 đơn trong phần học, đơn đó trễ. Nếu tính trong mẫu, r của dòng đó là bao nhiêu (theo p̄ của fold T11)? Tính chéo thì sao? Vì sao m = 10 vẫn không đủ để bỏ qua việc tính chéo?

## Bước 3 (cơ bản) · Vòng K-fold tự viết; r trong fold và ngoài fold

In [ ]:
def cv_tu_viet(X, y, cv, tao_mo_hinh):
    # TODO: vòng K-fold tự viết: với mỗi (a, b) trong cv.split(X, y): fit tao_mo_hinh() trên phần học a, tính AP trên phần giữ lại b;
    #       nếu phần giữ lại không có đơn trễ thì ghi None (không thay bằng 0). Trả danh sách K số.
    raise NotImplementedError

def cv_co_r(bang, cv, trong_fold=True):
    # TODO: LOG + r. trong_fold=True: r học lại trong từng fold (dùng tao_dac_trung). trong_fold=False: r tính MỘT lần trên cả `bang` rồi mới chia (cách sai, để so).
    raise NotImplementedError

In [ ]:
HOC11, _ = chia_thang(PHAT_TRIEN, '2017-11')                 # phần học của fold T11 (= train của Bài 4–6, nhưng thứ tự dòng khác)
cv = StratifiedKFold(5, shuffle=True, random_state=7)
tu_viet = cv_tu_viet(HOC11[FEATURES], HOC11.late, cv, LOG)
thu_vien = cross_validate(LOG(), HOC11[FEATURES], HOC11.late, cv=cv, scoring='average_precision')['test_score']
r_trong, r_ngoai = cv_co_r(HOC11, cv, True), cv_co_r(HOC11, cv, False)
print('LOG · tự viết', np.round(tu_viet, 4), '· thư viện', np.round(thu_vien, 4))
print('LOG + r · trong fold', round(np.mean(r_trong), 4), '· tính trước khi chia', round(np.mean(r_ngoai), 4))

In [ ]:
# TỰ KIỂM · bước 3
kiem('vòng tự viết trùng cross_validate từng fold', max(abs(a - b) for a, b in zip(tu_viet, thu_vien)) < 1e-12)
kiem('r tính trước khi chia cho CV cao hơn hẳn r trong fold', np.mean(r_ngoai) > np.mean(r_trong) + 0.05, 'r_ngoai phải dùng r tính trên cả HOC11')

## Bước 4 (cơ bản) · Đánh giá cuốn theo 8 tháng: LOG so với LOG + r, theo cặp

In [ ]:
def danh_gia_cuon(bang, cac_thang, m=M_LAM_TRON):
    # TODO: với mỗi tháng t: (hoc, cham) = chia_thang(bang, t); AP của LOG và của LOG + r (tao_dac_trung) trên cham.
    #       Trả DataFrame cột: thang, n_hoc, n_cham, ty_le, LOG, LOG_R, hieu (= LOG_R − LOG)
    raise NotImplementedError

In [ ]:
t0 = time.time()
CUON = danh_gia_cuon(PHAT_TRIEN, THANG)
THANG_R, THUA_R = int((CUON.hieu > 0).sum()), int((CUON.hieu < 0).sum())     # tháng hoà (hieu = 0) bị bỏ
P_DAU = kiem_dinh_dau(THANG_R, THANG_R + THUA_R)
print(CUON.round(4).to_string(index=False))
print(f'LOG + r thắng {THANG_R}, thua {THUA_R} · kiểm định dấu hai phía p = {P_DAU} ≈ {float(P_DAU):.3f} · {time.time() - t0:.0f} giây')

In [ ]:
# TỰ KIỂM · bước 4
kiem('8 tháng, đủ cột', len(CUON) == 8 and list(CUON.columns) == ['thang', 'n_hoc', 'n_cham', 'ty_le', 'LOG', 'LOG_R', 'hieu'])
kiem('cột LOG khớp slide (logistic từng tháng)', max(abs(a - b) for a, b in zip(CUON.LOG, [0.054512157666620965, 0.04188436841353713, 0.049056766695317, 0.04033175446246897, 0.08585197939253682, 0.07135903970349965, 0.18071780972837942, 0.0996692115328027])) < 1e-9)
kiem('hiệu = LOG_R − LOG', np.allclose(CUON.hieu, CUON.LOG_R - CUON.LOG))

**Câu hỏi.** Tháng nào LOG + r hơn LOG nhiều nhất, và kém nhất? Tỷ lệ trễ của các tháng đó thế nào? Vì sao không lấy trung bình AP qua 8 tháng để so?

## Bước 5 (khá) · Ghi quy tắc chọn, rồi chấm tập cuối một lần

Viết quy tắc chọn của bạn (chuỗi `QUY_TAC`) trong ô dưới **trước** khi chạy ô chấm cuối. `cham_cuoi(QUY_TAC, chon, tao_dac_trung)` ghi vào `.project_bai07_cham_cuoi.json` quy tắc, lựa chọn và dấu vân tay của dự đoán trên fold phát triển T11. Gọi lại y hệt nhận lại kết quả cũ; đổi quy tắc, đổi lựa chọn, hay giữ chuỗi mà đổi hàm tạo đặc trưng đều bị chặn. Đây là cơ chế nhắc kỷ luật, không phải bảo mật: giảng viên đối chiếu tệp dấu vết. Ô tự kiểm không cho biết đáp số của tập cuối.

In [ ]:
# TODO: viết quy tắc chọn của BẠN (chuỗi QUY_TAC) và áp nó để gán chon = 'LOG' hoặc 'LOG_R', dựa trên CUON, THANG_R, THUA_R, P_DAU.
#       Viết TRƯỚC khi chạy ô chấm cuối; sau khi chấm, không sửa ô này.
QUY_TAC = '...'
chon = ...
raise NotImplementedError

## Bước 6 (khá) · Chấm tập cuối và báo cáo một trang

In [ ]:
kq = cham_cuoi(QUY_TAC, chon, tao_dac_trung)
print(f"tập cuối: {kq['n']} đơn, tỷ lệ trễ {kq['ty_le']:.4f}; học trên {kq['n_hoc']} đơn 2017 đã giao trước 01/01/2018")
print(f"AP · LOG {kq['ap_LOG']:.4f} · LOG_R {kq['ap_LOG_R']:.4f} · cấu hình đã chọn: {kq['chon']}")
print(pd.DataFrame(kq['theo_thang']).round(4).to_string(index=False))

In [ ]:
# TỰ KIỂM · bước 5–6
kiem('đã ghi quy tắc và chọn LOG hoặc LOG_R', isinstance(QUY_TAC, str) and len(QUY_TAC) > 30 and chon in ('LOG', 'LOG_R'))
kiem('tập cuối 20.608 đơn, 3 tháng, AP trong [0; 1]', kq['n'] == 20608 and len(kq['theo_thang']) == 3 and 0 <= kq['ap_LOG'] <= 1 and 0 <= kq['ap_LOG_R'] <= 1)
vet = json.loads(_DAU_VET.read_text(encoding='utf-8'))
kiem('tệp dấu vết lưu đúng quy tắc đã viết và lựa chọn', vet['quy_tac'] == QUY_TAC and vet['chon'] == chon)
kiem('gọi lại y hệt nhận lại đúng kết quả cũ', cham_cuoi(QUY_TAC, chon, tao_dac_trung) == kq)
def bi_chan(*args):
    try:
        cham_cuoi(*args); return False
    except RuntimeError:
        return True
kiem('đổi lựa chọn thì bị chặn', bi_chan(QUY_TAC, 'LOG_R' if chon == 'LOG' else 'LOG', tao_dac_trung))
kiem('đổi quy tắc thì bị chặn', bi_chan(QUY_TAC + ' (sửa sau)', chon, tao_dac_trung))
kiem('giữ chuỗi nhưng đổi mô hình (m = 0) thì bị chặn', bi_chan(QUY_TAC, chon, lambda h, c: tao_dac_trung(h, c, m=0)))

## Báo cáo một trang (nộp cùng notebook), theo mẫu báo cáo có điều kiện

1. **Bài toán và thời điểm.** Dự đoán lúc duyệt; cột nào dùng được, giả định nào (bước 1).
2. **Quy trình.** r chỉ học từ phần học, r của dòng học tính chéo; phần học chỉ gồm đơn đã giao trước mốc; vòng K-fold tự viết khớp thư viện; r trong và ngoài fold khác nhau thế nào.
3. **Đánh giá.** Bảng 8 tháng, so theo cặp, kiểm định dấu, quy tắc chọn đã ghi; kết quả tập cuối (một lần), kèm tỷ lệ nền và AP từng tháng.
4. **Kết luận và giới hạn.** Câu trả lời đúng mức số liệu (“chưa phân biệt được” là một câu trả lời). Ít nhất ba giới hạn: nhãn cuối cùng; các tháng không độc lập; tập cuối thuộc test lịch sử; người bán của dòng hàng đầu; giả định thời điểm ghi cột.

## Nâng cao (chọn một; không có ô tự kiểm, chấm theo rubric)

- **A · Người bán mới:** chỉ chấm các đơn có người bán chưa có trong phần học. r còn giúp gì không?
- **B · Chọn m bằng đánh giá lồng theo thời gian:** chọn m trong {0; 1; 10; 100} trên các tháng trước, chấm tháng sau.
- **C · Dự đoán lúc giao hãng:** thêm số ngày đến lúc giao hãng; được thêm bao nhiêu AP, mất bao nhiêu ngày để can thiệp?

In [ ]:
# Nâng cao A (tuỳ chọn): người bán mới, chia theo nhóm kết hợp thời gian

In [ ]:
# Nâng cao B (tuỳ chọn): chọn m bằng đánh giá lồng theo thời gian

In [ ]:
# Nâng cao C (tuỳ chọn): dự đoán lúc giao hãng, đánh đổi với thời gian can thiệp